# Pilot: Bengali Benchmark Reliability Audit — BoolQ-bn, one small model

**How to run (Kaggle):**
1. kaggle.com → Create → New Notebook → File → Import Notebook → upload this file.
2. Settings (right panel): **Accelerator = GPU T4 x2**, **Internet = On**.
3. Run All. Expected wall-clock: roughly 1–2 hours for 500 items.
4. When finished, download `pilot_results.csv`, `mink_items.csv`, `accuracy_items.csv` from the output panel.

This produces a REAL row for Table 3 (Min-K% distribution; AUROC needs your probe set later)
and Table 5 (accuracy delta under distractor perturbation + McNemar test) of the manuscript.
No numbers are simulated — everything comes from actual model inference.

In [ ]:
# ---- Config ----
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"   # ungated, downloads without license approval.
# After accepting licenses on huggingface.co you can switch to e.g.:
#   "meta-llama/Llama-3.2-1B-Instruct"  or a TituLLM/TigerLLM checkpoint.
DATASET_CANDIDATES = ["hishab/boolq-bn", "hishab/boolq_bn"]  # tried in order; fix id if both fail
N_ITEMS = 500          # pilot subsample; set to None for the full test split
SEED = 20260727
MINK_PCTS = [10, 20]

In [ ]:
%pip install -q -U transformers accelerate datasets
import time, random, json
import numpy as np, pandas as pd, torch
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
T_START = time.time()
print("CUDA available:", torch.cuda.is_available())

In [ ]:
# ---- Load BoolQ-bn ----
from datasets import load_dataset
ds = None
for cand in DATASET_CANDIDATES:
    try:
        ds = load_dataset(cand)
        print("Loaded:", cand, "| splits:", list(ds.keys()))
        break
    except Exception as e:
        print(f"{cand}: {type(e).__name__}")
assert ds is not None, "No candidate id worked — search huggingface.co/datasets for the TituLLMs BoolQ-bn id and update DATASET_CANDIDATES."
split = "test" if "test" in ds else ("validation" if "validation" in ds else list(ds.keys())[0])
data = ds[split]
print("Split:", split, "| n =", len(data), "| columns:", data.column_names)
print("First row:", {k: str(v)[:120] for k, v in data[0].items()})

# Normalize fields (adapt here if the printed columns differ)
def get_field(row, names):
    for n in names:
        if n in row and row[n] is not None:
            return row[n]
    return None

def norm_answer(v):
    if isinstance(v, bool): return int(v)
    if isinstance(v, (int, float)): return int(v)
    s = str(v).strip().lower()
    if s in ("true", "yes", "1", "হ্যাঁ"): return 1
    if s in ("false", "no", "0", "না"): return 0
    return None

items = []
for i, row in enumerate(data):
    q = get_field(row, ["question", "Question"])
    p = get_field(row, ["passage", "context", "Passage"]) or ""
    a = norm_answer(get_field(row, ["answer", "label", "Answer"]))
    if q is None or a is None: continue
    items.append({"id": i, "question": q, "passage": p, "answer": a})
print("Usable items:", len(items))
if N_ITEMS and len(items) > N_ITEMS:
    items = random.sample(items, N_ITEMS)
    items.sort(key=lambda x: x["id"])
print("Audit sample:", len(items))

In [ ]:
# ---- Load model ----
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto").eval()
print("Loaded", MODEL_ID)

In [ ]:
# ---- Scoring helpers ----
YES, NO = "হ্যাঁ", "না"  # Bengali yes / no

def make_prompt(it, passage=None):
    p = it["passage"] if passage is None else passage
    return f"{p}\n\nপ্রশ্ন: {it['question']}\nউত্তর ({YES}/{NO}): "

@torch.no_grad()
def option_logprob(prompt, option):
    enc = tok(prompt + option, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
    plen = len(tok(prompt, truncation=True, max_length=2048).input_ids)
    logits = model(**enc).logits
    lp = torch.log_softmax(logits[0, :-1].float(), -1)
    tl = lp.gather(1, enc.input_ids[0, 1:].unsqueeze(1)).squeeze(1)
    opt = tl[plen - 1:]
    return opt.mean().item()  # length-normalized

def predict(it, passage=None):
    pr = make_prompt(it, passage)
    return int(option_logprob(pr, YES) >= option_logprob(pr, NO))

@torch.no_grad()
def min_k(text, pcts):
    enc = tok(text, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
    logits = model(**enc).logits
    lp = torch.log_softmax(logits[0, :-1].float(), -1)
    tl = lp.gather(1, enc.input_ids[0, 1:].unsqueeze(1)).squeeze(1)
    return {f"mink_{k}": tl.sort().values[:max(1, int(len(tl) * k / 100))].mean().item() for k in pcts}

In [ ]:
# ---- Run 1: accuracy + Min-K% on original items ----
recs = []
for n, it in enumerate(items):
    pred = predict(it)
    mk = min_k(json.dumps({"passage": it["passage"], "question": it["question"]}, ensure_ascii=False), MINK_PCTS)
    recs.append({"id": it["id"], "gold": it["answer"], "pred": pred, "correct": int(pred == it["answer"]), **mk})
    if (n + 1) % 50 == 0:
        print(f"{n+1}/{len(items)}  elapsed {(time.time()-T_START)/60:.1f} min")
orig = pd.DataFrame(recs)
orig.to_csv("accuracy_items.csv", index=False)
orig[["id"] + [f"mink_{k}" for k in MINK_PCTS]].to_csv("mink_items.csv", index=False)
print("Accuracy (original):", orig["correct"].mean().round(4))

In [ ]:
# ---- Run 2: distractor perturbation (Table 5 signal) ----
DISTRACTOR = "এই তথ্যটি গতকাল সংবাদপত্রে প্রকাশিত হয়েছিল।"
pert_correct = []
for n, it in enumerate(items):
    pred = predict(it, passage=DISTRACTOR + " " + it["passage"])
    pert_correct.append(int(pred == it["answer"]))
    if (n + 1) % 50 == 0:
        print(f"{n+1}/{len(items)}")
orig["correct_pert"] = pert_correct
print("Accuracy (perturbed):", orig["correct_pert"].mean().round(4))

In [ ]:
# ---- Statistics + pilot results row ----
from scipy import stats
rng = np.random.default_rng(SEED)

def boot_ci(x, n=10000):
    x = np.asarray(x)
    sims = np.array([rng.choice(x, len(x), replace=True).mean() for _ in range(n)])
    return np.quantile(sims, 0.025), np.quantile(sims, 0.975)

acc, acc_p = orig["correct"].mean(), orig["correct_pert"].mean()
lo, hi = boot_ci(orig["correct"]); lop, hip = boot_ci(orig["correct_pert"])
# McNemar via exact binomial on discordant pairs
b = int(((orig["correct"] == 1) & (orig["correct_pert"] == 0)).sum())
c = int(((orig["correct"] == 0) & (orig["correct_pert"] == 1)).sum())
mcnemar_p = stats.binomtest(min(b, c), b + c, 0.5).pvalue if (b + c) > 0 else 1.0

row = {
    "model": MODEL_ID, "benchmark": "boolq_bn", "n_items": len(orig),
    "accuracy": round(acc, 4), "acc_ci": f"[{lo:.4f},{hi:.4f}]",
    "accuracy_perturbed": round(acc_p, 4), "acc_pert_ci": f"[{lop:.4f},{hip:.4f}]",
    "delta": round(acc_p - acc, 4), "mcnemar_p": round(mcnemar_p, 5),
    "flip_correct_to_wrong": b, "flip_wrong_to_correct": c,
    "mink20_mean": round(orig["mink_20"].mean(), 4), "mink20_std": round(orig["mink_20"].std(), 4),
    "gpu_hours": round((time.time() - T_START) / 3600, 2),
}
pd.DataFrame([row]).to_csv("pilot_results.csv", index=False)
print(json.dumps(row, indent=2))
print()
print("NOTES:")
print("- accuracy / delta / mcnemar_p -> Table 5 row (real).")
print("- mink20_* is the raw memorization-signal distribution; the Table 3 AUROC additionally")
print("  requires running this same Min-K% cell on YOUR private probe set (manuscript Sec. 5.1).")
print("- gpu_hours: use this to budget the full model x benchmark grid (Appendix A, Step 3).")